# 🧪 Exercício Bônus — Transferência de Massa Multicomponente
## Formulação de Maxwell-Stefan: Tubo de Stefan Modificado

---

**Sistema:** Acetona (1) + Benzeno (2) + Metanol (3) evaporando através de um filme gasoso estagnado de Ar (4)

**Referência:** Taylor & Krishna, *Multicomponent Mass Transfer*, Wiley, 1993 — Exemplo 8.6.1

---

## 📐 Fundamentos Teóricos

### 1. Equações de Maxwell-Stefan

Para uma mistura de $n$ componentes em estado estacionário, a equação de Maxwell-Stefan para o componente $i$ é:

$$\frac{dx_i}{dz} = \sum_{\substack{j=1 \\ j \neq i}}^{n} \frac{x_j N_i - x_i N_j}{c_T \, \mathfrak{D}_{ij}}$$

onde:
- $x_i$ = fração molar do componente $i$
- $N_i$ = fluxo molar do componente $i$ $[\text{mol}/(\text{m}^2 \cdot \text{s})]$
- $c_T = P/(RT)$ = concentração molar total (gás ideal) $[\text{mol/m}^3]$
- $\mathfrak{D}_{ij}$ = coeficiente de difusão binário de Maxwell-Stefan $[\text{m}^2/\text{s}]$

### 2. Condição de Ar Estagnado ($N_4 = 0$)

Para $i = 1, 2, 3$ e separando o termo do Ar ($j=4$, com $N_4 = 0$):

$$\frac{dx_i}{dz} = \sum_{\substack{j=1 \\ j \neq i}}^{3} \frac{x_j N_i - x_i N_j}{c_T \, \mathfrak{D}_{ij}} + \frac{x_4 N_i}{c_T \, \mathfrak{D}_{i4}}$$

Reagrupando os termos em função dos fluxos $N_1, N_2, N_3$:

$$c_T \frac{dx_i}{dz} = N_i \underbrace{\left(\sum_{\substack{j=1 \\ j \neq i}}^{3} \frac{x_j}{\mathfrak{D}_{ij}} + \frac{x_4}{\mathfrak{D}_{i4}}\right)}_{B_{ii}} - \underbrace{x_i}_{} \sum_{\substack{j=1 \\ j \neq i}}^{3} \frac{N_j}{\mathfrak{D}_{ij}}$$

### 3. Formulação Matricial

O sistema pode ser escrito na forma compacta:

$$c_T \, \frac{d\mathbf{x}}{dz} = [B(\mathbf{x})] \, \mathbf{N}$$

onde $\mathbf{x} = [x_1, x_2, x_3]^T$, $\mathbf{N} = [N_1, N_2, N_3]^T$ e a matriz $[B]$ é **3×3** com:

$$\boxed{B_{ii} = \sum_{\substack{j=1 \\ j \neq i}}^{3} \frac{x_j}{\mathfrak{D}_{ij}} + \frac{x_4}{\mathfrak{D}_{i4}}, \qquad B_{ij} = -\frac{x_i}{\mathfrak{D}_{ij}} \quad (i \neq j)}$$

### 4. Estratégia de Solução — Método de Tiro (*Shooting Method*)

O problema é um **Problema de Valor de Contorno (PVC)**:
- **EDO:** $\dfrac{d\mathbf{x}}{dz} = \dfrac{1}{c_T} [B(\mathbf{x})] \, \mathbf{N}$
- **CI em $z=0$:** $x_1=0.319,\ x_2=0.150,\ x_3=0.350$
- **CC em $z=\delta$:** $x_1=0,\ x_2=0,\ x_3=0$

Como $\mathbf{N}$ é **constante** em estado estacionário (conservação de fluxo), usamos o **método de tiro**:
1. Chutamos $\mathbf{N}^{(k)}$
2. Integramos a EDO de $z=0$ até $z=\delta$
3. Calculamos o resíduo $\mathbf{F}(\mathbf{N}) = \mathbf{x}(\delta; \mathbf{N}) - [0,0,0]^T$
4. Usamos um método de Newton-Raphson para encontrar $\mathbf{N}$ tal que $\mathbf{F}=\mathbf{0}$

---
## ⚙️ Passo 0 — Importação de Bibliotecas

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, Markdown

# Configuração de estilo dos gráficos
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 120
})

print("✓ Bibliotecas carregadas com sucesso!")

---
## 📋 Passo 1 — Dados do Problema

In [ ]:
# ============================================================
# DADOS DO SISTEMA
# Componentes: 1=Acetona  2=Benzeno  3=Metanol  4=Ar (estagnado)
# ============================================================

# --- Condições operacionais ---
T     = 328.5     # Temperatura [K]
P     = 99.4e3    # Pressão [Pa]
R     = 8.314     # Constante universal dos gases [J/(mol·K)]
delta = 0.238     # Espessura do filme gasoso (caminho de difusão) [m]

# --- Concentração molar total (lei dos gases ideais) ---
cT = P / (R * T)  # [mol/m³]
print(f"Concentração molar total: cT = P/(RT) = {P:.1f}/({R}×{T}) = {cT:.4f} mol/m³")

# --- Coeficientes de difusão binária de Maxwell-Stefan [m²/s] ---
# (Ejemplo 8.6.1 de Taylor & Krishna para D13, D14, D34)
# (Equação de Fuller-Schettler-Giddings para D12, D23, D24)
D = {}  # Dicionário D[(i,j)] = Dij

D[(1,2)] = 5.04e-6    # Acetona  – Benzeno
D[(1,3)] = 8.48e-6    # Acetona  – Metanol
D[(1,4)] = 13.72e-6   # Acetona  – Ar
D[(2,3)] = 7.55e-6    # Benzeno  – Metanol
D[(2,4)] = 9.02e-6    # Benzeno  – Ar
D[(3,4)] = 19.91e-6   # Metanol  – Ar

# Simetria: D_ij = D_ji
for (i,j), val in list(D.items()):
    D[(j,i)] = val

print("\n--- Coeficientes de Difusão Binária [m²/s] ---")
pares = [(1,2,'Acetona-Benzeno'), (1,3,'Acetona-Metanol'), (1,4,'Acetona-Ar'),
         (2,3,'Benzeno-Metanol'), (2,4,'Benzeno-Ar'),    (3,4,'Metanol-Ar')]
for i,j,nome in pares:
    print(f"  Đ{i}{j} ({nome:20s}) = {D[(i,j)]*1e6:.2f} × 10⁻⁶ m²/s")

# --- Condições de contorno ---
# z = 0 : interface líquido-vapor
x0 = np.array([0.319, 0.150, 0.350])   # [x1, x2, x3]
x4_0 = 1.0 - x0.sum()

# z = delta : topo do tubo (vapores varridos pelo ar)
x_delta = np.array([0.0, 0.0, 0.0])    # [x1, x2, x3]
x4_delta = 1.0 - x_delta.sum()

print(f"\n--- Condições de Contorno ---")
print(f"  z = 0     : x1={x0[0]:.3f}  x2={x0[1]:.3f}  x3={x0[2]:.3f}  x4={x4_0:.3f}")
print(f"  z = δ     : x1={x_delta[0]:.3f}  x2={x_delta[1]:.3f}  x3={x_delta[2]:.3f}  x4={x4_delta:.3f}")
print(f"\n  Soma em z=0: {x0.sum() + x4_0:.4f} (deve ser 1.0)")

---
## 🔢 Passo 2 — Construção da Matriz [B(x)]

Para o nosso sistema com 4 componentes e $N_4 = 0$, a matriz **3×3** é:

$$[B] = \begin{bmatrix}
\dfrac{x_2}{\mathfrak{D}_{12}}+\dfrac{x_3}{\mathfrak{D}_{13}}+\dfrac{x_4}{\mathfrak{D}_{14}} & -\dfrac{x_1}{\mathfrak{D}_{12}} & -\dfrac{x_1}{\mathfrak{D}_{13}} \\[10pt]
-\dfrac{x_2}{\mathfrak{D}_{12}} & \dfrac{x_1}{\mathfrak{D}_{12}}+\dfrac{x_3}{\mathfrak{D}_{23}}+\dfrac{x_4}{\mathfrak{D}_{24}} & -\dfrac{x_2}{\mathfrak{D}_{23}} \\[10pt]
-\dfrac{x_3}{\mathfrak{D}_{13}} & -\dfrac{x_3}{\mathfrak{D}_{23}} & \dfrac{x_1}{\mathfrak{D}_{13}}+\dfrac{x_2}{\mathfrak{D}_{23}}+\dfrac{x_4}{\mathfrak{D}_{34}}
\end{bmatrix}$$

> **Observação:** A diagonal principal contém termos de **arraste** (interações do componente $i$ com todos os outros), enquanto os termos fora da diagonal capturam a **difusão cruzada** (a influência do gradiente de $x_j$ sobre o fluxo de $x_i$).

In [ ]:
def build_B_matrix(x):
    """
    Constrói a matriz [B] de Maxwell-Stefan para 4 componentes com N4=0.

    Parâmetros
    ----------
    x : array-like, shape (3,)
        Frações molares [x1, x2, x3]. A fração do Ar é x4 = 1 - x1 - x2 - x3.

    Retorna
    -------
    B : ndarray, shape (3, 3)
        Matriz de coeficientes (unidade: s/m²).
    """
    x1, x2, x3 = x[0], x[1], x[2]
    x4 = 1.0 - x1 - x2 - x3        # fração molar do Ar

    B = np.zeros((3, 3))

    # ------ Linha 1: equação de Maxwell-Stefan para Acetona (i=1) ------
    B[0, 0] =  x2/D[(1,2)] + x3/D[(1,3)] + x4/D[(1,4)]   # B_11 (diagonal)
    B[0, 1] = -x1/D[(1,2)]                                  # B_12 (difusão cruzada 1←2)
    B[0, 2] = -x1/D[(1,3)]                                  # B_13 (difusão cruzada 1←3)

    # ------ Linha 2: equação de Maxwell-Stefan para Benzeno (i=2) ------
    B[1, 0] = -x2/D[(1,2)]                                  # B_21 (difusão cruzada 2←1)
    B[1, 1] =  x1/D[(1,2)] + x3/D[(2,3)] + x4/D[(2,4)]   # B_22 (diagonal)
    B[1, 2] = -x2/D[(2,3)]                                  # B_23 (difusão cruzada 2←3)

    # ------ Linha 3: equação de Maxwell-Stefan para Metanol (i=3) ------
    B[2, 0] = -x3/D[(1,3)]                                  # B_31 (difusão cruzada 3←1)
    B[2, 1] = -x3/D[(2,3)]                                  # B_32 (difusão cruzada 3←2)
    B[2, 2] =  x1/D[(1,3)] + x2/D[(2,3)] + x4/D[(3,4)]   # B_33 (diagonal)

    return B


# ---- Verificação: calcular [B] nas condições de interface (z=0) ----
B_interface = build_B_matrix(x0)

print("Matriz [B] calculada em z=0 (interface líquida):")
print(f"  x = {x0},  x4 = {1-x0.sum():.3f}\n")
print("  [B] (s/m²) =\n")
for i in range(3):
    row = "  ["
    for j in range(3):
        row += f" {B_interface[i,j]*1e6:10.4f}"
    row += "  ] × 10⁻⁶"
    print(row)

print(f"\n  det([B]) = {np.linalg.det(B_interface):.4e} s³/m⁶")
print("  (det ≠ 0: sistema é inversível ✓)")

---
## 🔄 Passo 3 — Sistema de Equações Diferenciais Ordinárias

A EDO a integrar é:

$$\frac{d\mathbf{x}}{dz} = \frac{1}{c_T} [B(\mathbf{x})] \, \mathbf{N}$$

com $c_T = P/(RT)$ e $\mathbf{N} = [N_1, N_2, N_3]^T$ **constante** ao longo de $z$ (estado estacionário).

> **Por que N é constante?** Em estado estacionário, o balanço de massa de cada componente no filme exige que $dN_i/dz = 0$, portanto $N_i = \text{cte}$ em todo o filme.

In [ ]:
def sistema_ode(z, x, N):
    """
    Sistema de EDOs das equações de Maxwell-Stefan.

    Calcula dx/dz = (1/cT) * [B(x)] @ N

    Parâmetros
    ----------
    z : float
        Posição ao longo do filme [m] (variável independente).
    x : array, shape (3,)
        Frações molares atuais [x1, x2, x3].
    N : array, shape (3,)
        Fluxos molares [N1, N2, N3] em mol/(m²·s) — constantes.

    Retorna
    -------
    dxdz : array, shape (3,)
        Derivadas das frações molares.
    """
    B = build_B_matrix(x)          # monta [B] no ponto atual
    dxdz = (1.0 / cT) * (B @ N)   # produto matriz-vetor
    return dxdz


# ---- Teste da EDO com um chute razoável de N ----
N_teste = np.array([1e-3, 5e-4, 1.5e-3])  # chute arbitrário [mol/(m²·s)]
dxdz_teste = sistema_ode(0.0, x0, N_teste)

print("Teste do sistema de EDOs em z=0 com N arbitrário:")
print(f"  N  = {N_teste} mol/(m²·s)")
print(f"  dx/dz = {dxdz_teste} m⁻¹")
print("\n  (Sinal positivo = fração molar aumentando em z; negativo = diminuindo)")

---
## 🎯 Passo 4 — Método de Tiro (*Shooting Method*)

### Formulação do Problema de Otimização

Queremos encontrar $\mathbf{N} = [N_1, N_2, N_3]^T$ tal que, ao integrar a EDO de $z=0$ até $z=\delta$, as condições de contorno em $z=\delta$ sejam satisfeitas:

$$\mathbf{F}(\mathbf{N}) = \mathbf{x}(\delta; \mathbf{N}) - \mathbf{x}_{\delta} = \mathbf{0}$$

Usamos `scipy.optimize.fsolve` (método de Newton-Raphson com diferenças finitas para o Jacobiano).

In [ ]:
def integrar_ode(N_guess):
    """
    Integra o sistema de EDOs de z=0 até z=delta para um dado chute de N.

    Parâmetros
    ----------
    N_guess : array, shape (3,)
        Chute para os fluxos molares [N1, N2, N3].

    Retorna
    -------
    sol : OdeResult
        Objeto de solução do solve_ivp (contém t, y, e interpolação densa).
    """
    N = np.array(N_guess)
    sol = solve_ivp(
        fun=lambda z, x: sistema_ode(z, x, N),
        t_span=(0.0, delta),
        y0=x0,
        method='RK45',       # Runge-Kutta de 4ª/5ª ordem (Dormand-Prince)
        rtol=1e-9,           # tolerância relativa
        atol=1e-11,          # tolerância absoluta
        dense_output=True    # permite interpolar a solução em qualquer z
    )
    return sol


def residuos(N_guess):
    """
    Função de resíduos para o método de tiro.

    Compara x(delta) calculado com a condição de contorno desejada [0, 0, 0].

    Parâmetros
    ----------
    N_guess : array, shape (3,)
        Chute para os fluxos molares.

    Retorna
    -------
    F : array, shape (3,)
        Resíduo F = x(delta) - x_delta (deve convergir para [0,0,0]).
    """
    sol = integrar_ode(N_guess)
    x_final = sol.y[:, -1]           # x em z = delta
    return x_final - x_delta          # resíduo


# ============================================================
# Estimativa inicial para os fluxos (chute inicial do método de tiro)
# Usa lei de Fick simplificada: N_i ≈ cT * D_i4 * (x_i(0) - x_i(δ)) / δ
# ============================================================
D_i4 = np.array([D[(1,4)], D[(2,4)], D[(3,4)]])
N_inicial = cT * D_i4 * (x0 - x_delta) / delta

print("Chute inicial (Fick binário simplificado):")
nomes = ['Acetona (N₁)', 'Benzeno (N₂)', 'Metanol  (N₃)']
for nome, ni in zip(nomes, N_inicial):
    print(f"  {nome} = {ni:.4e} mol/(m²·s)")

print("\nResíduo com chute inicial:")
res_inicial = residuos(N_inicial)
for i, r in enumerate(res_inicial):
    print(f"  F{i+1} = {r:.6f}  (≠ 0 → ainda não convergiu)")

---
## ✅ Passo 5 — Resolução e Obtenção dos Fluxos Molares

In [ ]:
print("Resolvendo o sistema não-linear pelo método de Newton-Raphson...")
print("(fsolve com Jacobiano por diferenças finitas)\n")

N_solucao, info, ier, mensagem = fsolve(
    residuos,
    N_inicial,
    full_output=True,
    xtol=1e-12,
    ftol=1e-12
)

# Verificação de convergência
if ier == 1:
    print("✓ Convergência atingida!")
else:
    print(f"⚠ Atenção: {mensagem}")

# Resíduo final
res_final = residuos(N_solucao)
print(f"\nResíduo final ||F|| = {np.linalg.norm(res_final):.2e}")
print(f"  F1 = {res_final[0]:.2e}")
print(f"  F2 = {res_final[1]:.2e}")
print(f"  F3 = {res_final[2]:.2e}")

print("\n" + "="*55)
print("  FLUXOS MOLARES DE EVAPORAÇÃO — RESULTADOS FINAIS")
print("="*55)
nomes_comp = ['Acetona ', 'Benzeno ', 'Metanol ']
for i, (nome, ni) in enumerate(zip(nomes_comp, N_solucao)):
    print(f"  N{i+1} ({nome}) = {ni:+.6e} mol/(m²·s)")
print("="*55)
print("  Fluxo total N_T = N1+N2+N3 =",
      f"{N_solucao.sum():+.6e} mol/(m²·s)")
print("="*55)

---
## 🔍 Passo 6 — Análise da Solução: Verificação e Perfis

In [ ]:
# Integração final com a solução convergida
sol_final = integrar_ode(N_solucao)

# Pontos para visualização
z_vals = np.linspace(0, delta, 1000)
x_vals = sol_final.sol(z_vals)   # interpolação densa: shape (3, 1000)
x4_vals = 1.0 - x_vals.sum(axis=0)

print("Verificação das condições de contorno na solução final:")
print(f"  z = 0 m    → x = {sol_final.sol(0.0).round(4)}  (esperado: {x0})")
print(f"  z = δ={delta} m → x = {sol_final.sol(delta).round(6)}  (esperado: {x_delta})")

print("\n--- Variação das frações molares ao longo do filme ---")
print(f"  {'Componente':15s}  {'z=0':>10s}  {'z=δ/2':>10s}  {'z=δ':>10s}")
print("-" * 52)
componentes = ['Acetona (1)', 'Benzeno (2)', 'Metanol (3)', 'Ar      (4)']
for k, comp in enumerate(componentes):
    if k < 3:
        v0   = sol_final.sol(0.0)[k]
        vmid = sol_final.sol(delta/2)[k]
        vend = sol_final.sol(delta)[k]
    else:
        v0   = 1 - sol_final.sol(0.0).sum()
        vmid = 1 - sol_final.sol(delta/2).sum()
        vend = 1 - sol_final.sol(delta).sum()
    print(f"  {comp:15s}  {v0:10.4f}  {vmid:10.4f}  {vend:10.4f}")

---
## 📊 Passo 7 — Visualização dos Resultados

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

cores   = ['steelblue', 'tomato', 'mediumseagreen', 'gray']
labels  = ['Acetona (1)', 'Benzeno (2)', 'Metanol (3)', 'Ar (4)']
lstyles = ['-', '-', '-', '--']
z_cm    = z_vals * 100  # m → cm

# --------------------------------------------------
# Gráfico 1: Perfis de fração molar
# --------------------------------------------------
ax1 = fig.add_subplot(gs[0, :])

for k in range(3):
    ax1.plot(z_cm, x_vals[k], color=cores[k], ls=lstyles[k],
             lw=2.5, label=labels[k])
ax1.plot(z_cm, x4_vals, color=cores[3], ls='--', lw=2.0, label=labels[3])

# Marcadores nas condições de contorno
ax1.plot(0,         x0[0], 'o', color=cores[0], ms=8, zorder=5)
ax1.plot(0,         x0[1], 'o', color=cores[1], ms=8, zorder=5)
ax1.plot(0,         x0[2], 'o', color=cores[2], ms=8, zorder=5)
ax1.plot(delta*100, 0.0,   'o', color=cores[0], ms=8, zorder=5)
ax1.plot(delta*100, 0.0,   'o', color=cores[1], ms=8, zorder=5)
ax1.plot(delta*100, 0.0,   'o', color=cores[2], ms=8, zorder=5)

ax1.axvline(x=0,         color='k', ls=':', lw=1, alpha=0.5)
ax1.axvline(x=delta*100, color='k', ls=':', lw=1, alpha=0.5)
ax1.text(0.5,  0.97, 'Interface\nlíquida\n(z = 0)', ha='center', va='top',
         transform=ax1.get_xaxis_transform(), fontsize=9, color='gray')
ax1.text(delta*100 - 0.5, 0.97, 'Topo do tubo\n(z = δ)', ha='center', va='top',
         transform=ax1.get_xaxis_transform(), fontsize=9, color='gray')

ax1.set_xlabel('Posição z [cm]')
ax1.set_ylabel('Fração Molar $x_i$')
ax1.set_title('Perfis de Composição no Filme Gasoso — Maxwell-Stefan Multicomponente')
ax1.legend(loc='center right')
ax1.set_xlim([0, delta*100])
ax1.set_ylim([-0.02, 1.02])
ax1.grid(True, alpha=0.3)

# --------------------------------------------------
# Gráfico 2: Fluxos molares (barras)
# --------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0])

comp_labels = ['Acetona\n$N_1$', 'Benzeno\n$N_2$', 'Metanol\n$N_3$']
N_plot = N_solucao * 1e4  # escala para × 10⁻⁴
bars = ax2.bar(comp_labels, N_plot, color=cores[:3], edgecolor='black',
               linewidth=1.2, width=0.5)

for bar, val in zip(bars, N_plot):
    offset = 0.003 * (1 if val >= 0 else -1)
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + offset,
             f'{val:.3f}',
             ha='center', va='bottom' if val >= 0 else 'top',
             fontsize=11, fontweight='bold')

ax2.axhline(y=0, color='k', lw=0.8)
ax2.set_ylabel('$N_i \\times 10^{4}$ [mol/(m²·s)]')
ax2.set_title('Fluxos Molares de Evaporação')
ax2.grid(True, alpha=0.3, axis='y')

# --------------------------------------------------
# Gráfico 3: Gradientes de fração molar (dx/dz)
# --------------------------------------------------
ax3 = fig.add_subplot(gs[1, 1])

# Calcula dx/dz numericamente para cada ponto
dxdz_vals = np.array([
    sistema_ode(z_vals[k], x_vals[:, k], N_solucao)
    for k in range(len(z_vals))
]).T  # shape (3, n_pts)

for k in range(3):
    ax3.plot(z_cm, dxdz_vals[k], color=cores[k], lw=2.0, label=labels[k])

ax3.axhline(y=0, color='k', lw=0.8)
ax3.set_xlabel('Posição z [cm]')
ax3.set_ylabel('$dx_i/dz$ [m⁻¹]')
ax3.set_title('Gradientes de Fração Molar ao Longo do Filme')
ax3.legend()
ax3.set_xlim([0, delta*100])
ax3.grid(True, alpha=0.3)

plt.suptitle('Transferência de Massa Multicomponente — Equações de Maxwell-Stefan',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('maxwell_stefan_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figura salva como 'maxwell_stefan_resultados.png'")

---
## 🔬 Passo 8 — Análise dos Efeitos de Difusão Cruzada

Uma das principais motivações do experimento é verificar os **efeitos de difusão cruzada** (*cross-diffusion*). Para isso, comparamos o resultado Maxwell-Stefan com a predição da lei de Fick binária efetiva (sem difusão cruzada).

In [ ]:
# ============================================================
# Comparação: Maxwell-Stefan vs. Fick Binário (sem difusão cruzada)
# ============================================================

def D_efetivo_wilke(x, D_dict, n_comp=4):
    """
    Calcula a difusividade efetiva pelo método de Wilke para cada componente i
    em uma mistura (usado como referência para comparação com MS).

    D_e,i = (1 - x_i) / sum_{j≠i} (x_j / D_ij)
    """
    x_full = np.append(x, 1.0 - x.sum())  # inclui Ar
    D_eff = np.zeros(3)
    for i in range(1, 4):
        soma = sum(
            x_full[j-1] / D_dict[(i, j)]
            for j in range(1, n_comp+1)
            if j != i
        )
        D_eff[i-1] = (1.0 - x_full[i-1]) / soma if soma > 0 else 0.0
    return D_eff


# Fick binário (simplificado, Wilke): N_i = cT * D_eff * (x_i(0) - x_i(δ)) / δ
D_eff_0     = D_efetivo_wilke(x0, D)           # difusividade efetiva em z=0
D_eff_delta = D_efetivo_wilke(x_delta + 1e-9, D)  # em z=δ (evitar divisão por zero)
D_eff_media = (D_eff_0 + D_eff_delta) / 2

N_fick = cT * D_eff_media * (x0 - x_delta) / delta

# Exibição comparativa
print("┌─────────────────────────────────────────────────────────────┐")
print("│    COMPARAÇÃO: Maxwell-Stefan vs. Fick Efetivo (Wilke)      │")
print("├──────────────┬─────────────────┬─────────────────┬──────────┤")
print("│ Componente   │ N_MS [mol/m²s]  │ N_Fick [mol/m²s]│ Erro [%] │")
print("├──────────────┼─────────────────┼─────────────────┼──────────┤")

for i, nome in enumerate(['Acetona (1)', 'Benzeno (2)', 'Metanol (3)']):
    N_ms = N_solucao[i]
    N_f  = N_fick[i]
    erro = abs(N_ms - N_f) / abs(N_ms) * 100 if N_ms != 0 else 0
    print(f"│ {nome:12s} │ {N_ms:+.4e}    │ {N_f:+.4e}    │ {erro:7.1f}% │")

print("└──────────────┴─────────────────┴─────────────────┴──────────┘")
print()
print("Nota: Diferenças > 5% indicam efeitos significativos de difusão cruzada.")
print("A formulação de Maxwell-Stefan captura essas interações de forma exata.")

# Análise da matriz [B] para demonstrar difusão cruzada
print("\n--- Termos de difusão cruzada na interface (z=0) ---")
print("  Os termos off-diagonal de [B] quantificam o acoplamento entre componentes:")
nomes_b = ['Acetona', 'Benzeno', 'Metanol']
B0 = build_B_matrix(x0)
for i in range(3):
    for j in range(3):
        if i != j:
            print(f"  B[{i+1},{j+1}] ({nomes_b[i]}←{nomes_b[j]}) = {B0[i,j]*1e6:.4f} × 10⁻⁶ s/m²")

---
## 📝 Resumo Final

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║         RESUMO — EXERCÍCIO BÔNUS: TRANSFERÊNCIA DE MASSA    ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║ Sistema: Acetona(1)+Benzeno(2)+Metanol(3) / Ar(4) estagnado ║")
print(f"║ T = {T} K    P = {P/1e3} kPa    δ = {delta} m            ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║             Fluxos Molares de Evaporação                    ║")
print("╠══════════════════════════════════════════════════════════════╣")
for i, (nome, ni) in enumerate(zip(
        ['Acetona (N₁)', 'Benzeno (N₂)', 'Metanol  (N₃)'], N_solucao)):
    print(f"║  N{i+1} — {nome}: {ni:+.4e} mol/(m²·s)   ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Fluxo total N_T = {N_solucao.sum():+.4e} mol/(m²·s)           ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  Método: Maxwell-Stefan matricial + Shooting Method (RK45) ║")
print(f"║  Convergência: ||F|| = {np.linalg.norm(residuos(N_solucao)):.1e}                       ║")
print("╚══════════════════════════════════════════════════════════════╝")

---
## 💡 Conclusões

1. **Formulação matricial** de Maxwell-Stefan transforma $n-1$ equações acopladas em uma EDO vetorial: $d\mathbf{x}/dz = (1/c_T)[B(\mathbf{x})]\mathbf{N}$, onde a matriz $[B]$ depende das composições locais.

2. **Método de tiro** converte o problema de valor de contorno (PVC) em um sistema não-linear de $n-1$ equações nos fluxos $\mathbf{N}$, resolvido por Newton-Raphson.

3. **Efeitos de difusão cruzada** são capturados pelos termos off-diagonal de $[B]$: o gradiente de $x_j$ afeta o fluxo de $x_i$ mesmo quando $\nabla x_i = 0$ (fenômeno impossível na lei de Fick simples).

4. **Ar estagnado** ($N_4 = 0$) não aparece como variável independente, mas influencia todas as equações através de $x_4 = 1 - \sum_{i=1}^{3} x_i$ nos termos diagonais de $[B]$.

---
*Referência: Taylor & Krishna, Multicomponent Mass Transfer, Wiley (1993) — Exemplo 8.6.1*